In [3]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [4]:
try:
    import pypdf
    print(f"pypdf: {pypdf.__version__}")
except ImportError:
    print("pypdf 미설치 → pip install pypdf")

pypdf: 6.9.2


In [5]:
key = os.getenv("PINECONE_API_KEY")
print(key[:10])

pcsk_A4LsX


In [6]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=os.environ['PINECONE_API_KEY'])

In [7]:
INDEX_NAME = "finance-bok"
NAMESPACE  = "bok-ns1"

# 기존 인덱스 없으면 생성
if INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=INDEX_NAME,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print(f"{INDEX_NAME} 생성 완료")
else:
    print(f"{INDEX_NAME} 이미 존재함")


finance-bok 이미 존재함


In [8]:
# 상태 확인
f_index = pc.Index(INDEX_NAME)
print(f_index .describe_index_stats())

c:\Users\Admin\miniconda3\envs\langchain_rag_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'bok-ns1': {'vector_count': 249}},
 'total_vector_count': 249,
 'vector_type': 'dense'}


In [30]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "data/2026년 2월 경제전망보고서(Indigo Book)_FFF.pdf"

loader = PyPDFLoader(PDF_PATH)
docs = loader.load()

In [10]:
print(f"총 페이지 수: {len(docs)}")
print(f"\n첫 페이지 내용 (앞 300자):\n{docs[0].page_content[:300]}")
print(f"\n메타데이터: {docs[0].metadata}")

총 페이지 수: 93

첫 페이지 내용 (앞 300자):
경제전망 
 Indigo Book 
2026년 2월 
 
 
 
 
성장 2%대 반등, 부문별 온도차

메타데이터: {'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-02-26T16:48:20+09:00', 'author': 'A11', 'moddate': '2026-03-31T15:12:50+09:00', 'source': 'data/2026년 2월 경제전망보고서(Indigo Book)_FFF.pdf', 'total_pages': 93, 'page': 0, 'page_label': '1'}


In [11]:
# 출력 예시
{
    'source': 'data/2026년 2월 경제전망보고서(Indigo Book)_FFF.pdf',
    'page': 0
}

{'source': 'data/2026년 2월 경제전망보고서(Indigo Book)_FFF.pdf', 'page': 0}

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)

splits = splitter.split_documents(docs) #원본 문서의 메타 정보를 가져와 설정

In [13]:
print(f"총 청크 수: {len(splits)}")
print(f"\n첫 번째 청크:\n{splits[0].page_content}")
print(f"\n메타데이터: {splits[0].metadata}")
# 출력 예시
{
    'source': 'data/2026년 2월 경제전망보고서(Indigo Book)_FFF.pdf',
    'page': 0   # 원본 페이지 번호 유지
}

총 청크 수: 249

첫 번째 청크:
경제전망 
 Indigo Book 
2026년 2월 
 
 
 
 
성장 2%대 반등, 부문별 온도차

메타데이터: {'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-02-26T16:48:20+09:00', 'author': 'A11', 'moddate': '2026-03-31T15:12:50+09:00', 'source': 'data/2026년 2월 경제전망보고서(Indigo Book)_FFF.pdf', 'total_pages': 93, 'page': 0, 'page_label': '1'}


{'source': 'data/2026년 2월 경제전망보고서(Indigo Book)_FFF.pdf', 'page': 0}

In [14]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

INDEX_NAME = "finance-bok"
NAMESPACE  = "bok-ns1"

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")


In [15]:
print("업서트 시작...")
vectorstore = PineconeVectorStore.from_documents(
    documents=splits,
    embedding=embedding_model ,
    index_name=INDEX_NAME,
    namespace=NAMESPACE
)
print(f"업서트 완료 — 총 {len(splits)}개 청크")

업서트 시작...
업서트 완료 — 총 249개 청크


In [16]:
from pinecone import Pinecone

pc = Pinecone(api_key=os.environ['PINECONE_API_KEY'])
index = pc.Index(INDEX_NAME)
stats = index.describe_index_stats()
print(stats)

{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'bok-ns1': {'vector_count': 498}},
 'total_vector_count': 498,
 'vector_type': 'dense'}


In [17]:
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = PineconeVectorStore.from_existing_index(
    index_name=INDEX_NAME,
    embedding=embedding_model,
    namespace=NAMESPACE
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3, "namespace": NAMESPACE}
)

In [18]:
# 검색 테스트
queries = [
    "2026년 경제성장률 전망은?",
    "물가 상승률 전망은?",
    "고용 시장 전망은?"
]

In [19]:

for query in queries:
    print(f"[ 질문 ] {query}")
    
    results = retriever.invoke(query)
    
    for i, r in enumerate(results):
        print(f"  {i+1}. (p.{r.metadata.get('page', '')}) {r.page_content[:100]}")
    print()

[ 질문 ] 2026년 경제성장률 전망은?
  1. (p.43.0) 30 
 
<경제성장 전망1)> 
(전년동기대비, %) 
  2024 2025 2026e) 2027e) 
연간 상반 하반 연간 상반 하반 연간 연간 
GDP 성장률 2.0 0.3 
  2. (p.35.0) 22 
 
2. 거시경제 전망 
 
 
 
 
경제성장 
 
2-1. 금년중 국내경제는 美관세 영향, 건설투자의 더딘 회복에도 불구하고 반도체 경기 
개선세 확대, 예상보다 양호한
  3. (p.6.0) < 요약 1/8 > 
 
  
경제전망 요약 
 올해 우리 경제는 美관세 영향과 건설투자의 더딘 회복에도 불구하고 반도체 경기 
개선세 확대, 예상보다 양호한 세계경제 흐름 등에

[ 질문 ] 물가 상승률 전망은?
  1. (p.53.0) 1.9% 대비 0.1%p 높아졌다. 소비자물가 상승률의 경우에도 11월 전망1.9% 대비 0.1%p 높
아진 2.0%로 나타났다. 
 
시장의 국내 성장 및 물가 전망 모두 올해 
  2. (p.53.0) 40 
 
3. 전망의 리스크 평가 
    
주요 리스크 요인 
 
3-1. 향후 성장 전망경로에는 반도체 경기, 글로벌 통상환경, 국제금융시장 등과 관련
한 불확실성이 크며, 
  3. (p.11.0) < 요약 6/8 > 
6  
  
전망의 리스크 
 
 향후 성장 전망경로에는 반도체 경기, 글로벌 통상환경, 국제금융시장 등과 관련한 불확
실성이 크며, 물가의 경우 유가, 환

[ 질문 ] 고용 시장 전망은?
  1. (p.10.0) ▪ 상품수지는 반도체가격의 큰 폭 상승 등으로 흑자규모가 크게 늘어날 전망이다. 
서비스수지는 경기회복에 따른 산업서비스특허사용료 등 수요 증가, 디지털서비스플랫폼 구
독료 등 지
  2. (p.10.0) ▪ 상품수지는 반도체가격의 큰 폭 상승 등으로 흑자규모가 크게 늘어날 전망이다. 
서비스수지는 경기회복에 따른 산업서비스특허사용료 등 수요 증가, 디지털서비스플랫폼 구
독료 등 지
  3. (p.35.0) 22 


In [20]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_openai import ChatOpenAI

In [21]:
llm    = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

In [22]:
prompt = ChatPromptTemplate.from_template("""
당신은 한국은행 경제전망 보고서를 기반으로 답변하는 금융 전문 어시스턴트입니다.
아래 참고 문서를 바탕으로 질문에 정확하게 답하세요.
문서에 없는 내용은 "보고서에서 확인되지 않습니다"라고 답하세요.

[참고문서]
{context}

[질문]
{question}

한글로 간결하고 정확하게 답변하세요.
""")

In [23]:
rag_chain = (
    RunnableParallel(
        context=retriever,
        question=RunnablePassthrough()
    )
    | prompt
    | llm
    | parser
)

In [24]:
question =  "수출 전망은 어떻게 되나요?"
rag_chain.invoke(question)

'수출은 견조한 흐름을 이어갈 것으로 전망됩니다.'

In [25]:
# RAG 테스트
questions = [
    "2026년 GDP 성장률 전망치는 얼마인가요?",
    "소비자물가 상승률은 어떻게 전망하나요?",
    "수출 전망은 어떻게 되나요?"
]

for q in questions:
    print(f"[ Q ] {q}")
    answer = rag_chain.invoke(q)
    print(f"[ A ] {answer}")
    print()

[ Q ] 2026년 GDP 성장률 전망치는 얼마인가요?
[ A ] 2026년 GDP 성장률 전망치는 2.0%입니다.

[ Q ] 소비자물가 상승률은 어떻게 전망하나요?
[ A ] 소비자물가 상승률은 2026년 2.0%로 전망됩니다. 이는 11월 전망인 1.9% 대비 0.1%p 높아진 수치입니다.

[ Q ] 수출 전망은 어떻게 되나요?
[ A ] 수출은 견조한 흐름을 이어갈 것으로 전망됩니다.



In [26]:
question = "반도체 경기 전망은 어떻게 되나요?"
rag_chain.invoke(question)

'반도체 경기 전망은 글로벌 반도체 매출이 지난해 4/4분기 중 가파른 증가세를 지속하고 있으며, 반도체 가격이 범용 반도체 중심으로 급등하고 있는 상황입니다. 또한, 반도체 기업의 생산능력이 수요 대비 완만히 확대되며 재고가 큰 폭으로 감소하고 있는 것으로 평가됩니다.'

In [ ]:
questions = [
    "반도체 경기 전망은?",
    "건설투자 전망은?",
    "환율 관련 리스크는 무엇인가요?",
    "미국 관세가 한국 경제에 미치는 영향은?"
]

for q in questions:
    print(f"[ Q ] {q}")
    print(f"[ A ] {rag_chain.invoke(q)}")
    print()

[ Q ] 반도체 경기 전망은?
[ A ] 반도체 경기 전망은 글로벌 반도체 매출이 지난해 4/4분기 중 가파른 증가세를 지속하고 있으며, 반도체 가격이 범용 반도체 중심으로 급등하고 있습니다. 또한, 반도체 기업의 생산능력이 수요 대비 완만히 확대되며 재고가 큰 폭으로 감소하고 있는 상황입니다.

[ Q ] 건설투자 전망은?
[ A ] 건설투자는 AI 관련 투자와 SOC 투자 증가로 부진이 완화될 전망이나, 주거용 건설의 부진이 지속되어 지난 전망경로를 하회할 것으로 예상됩니다. 지난해 건설투자는 9.9% 감소하였고, 금년 중 1.0% 증가할 것으로 보이며, 내년에는 1.5%의 더딘 회복세를 보일 전망입니다.

[ Q ] 환율 관련 리스크는 무엇인가요?
[ A ] 환율 관련 리스크는 고환율 지속과 정부의 물가안정대책 강화입니다.

[ Q ] 미국 관세가 한국 경제에 미치는 영향은?
[ A ] 보고서에서 확인되지 않습니다.



In [28]:
# RAG 답변 vs 일반 LLM 답변 비교
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

llm_base = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

questions = [
    "2026년 GDP 성장률 전망치는 얼마인가요?",
    "소비자물가 상승률은 어떻게 전망하나요?",
    "수출 전망은 어떻게 되나요?"
]

for q in questions:
    print(f"[ Q ] {q}")

    # 일반 LLM (RAG 없음)
    base_answer = parser.invoke(llm_base.invoke(q))
    print(f"[ 일반 LLM ] {base_answer[:150]}")

    # RAG 기반 LLM
    rag_answer = rag_chain.invoke(q)
    print(f"[ RAG 답변 ] {rag_answer[:150]}")
    print()

[ Q ] 2026년 GDP 성장률 전망치는 얼마인가요?
[ 일반 LLM ] 2026년의 GDP 성장률 전망치는 여러 경제 기관과 연구소에 따라 다를 수 있으며, 특정한 수치를 제공하기 위해서는 최신 경제 보고서나 예측 자료를 참조해야 합니다. 일반적으로 이러한 전망치는 경제 상황, 정책 변화, 글로벌 경제 동향 등에 따라 변동성이 크기 때문에
[ RAG 답변 ] 2026년 GDP 성장률 전망치는 2.0%입니다.

[ Q ] 소비자물가 상승률은 어떻게 전망하나요?
[ 일반 LLM ] 소비자물가 상승률 전망은 여러 요인에 따라 달라질 수 있습니다. 일반적으로 경제 성장률, 통화 정책, 원자재 가격, 공급망 문제, 그리고 글로벌 경제 상황 등이 주요한 영향을 미칩니다. 

2023년의 경우, 많은 국가들이 인플레이션 압력을 경험하고 있으며, 이는 에너
[ RAG 답변 ] 소비자물가 상승률은 2026년 1월 중 2.0%로 나타났으며, 이는 물가목표 수준인 2.0%까지 낮아진 것입니다.

[ Q ] 수출 전망은 어떻게 되나요?
[ 일반 LLM ] 수출 전망은 여러 요인에 따라 달라질 수 있습니다. 일반적으로 경제 성장률, 글로벌 수요, 환율 변동, 무역 정책, 그리고 특정 산업의 경쟁력 등이 중요한 요소로 작용합니다. 

2023년의 경우, 세계 경제의 회복세와 함께 일부 국가에서는 수출이 증가할 것으로 예상되
[ RAG 답변 ] 수출은 견조한 흐름을 이어갈 것으로 전망됩니다.



In [29]:
test_questions = [
    # 수치 확인용
    "2026년 GDP 성장률 전망치는 얼마인가요?",
    "2026년 소비자물가 상승률 전망치는?",
    "2026년 수출 전망 금액은 얼마인가요?",

    # 범위 밖 질문 (환각 테스트)
    "2030년 경제성장률 전망은?",
    "미국 연방준비제도의 금리 결정 일정은?",

    # 맥락 이해 확인
    "성장률이 2%대로 반등하는 주요 원인은?",
    "부문별 온도차가 발생하는 이유는?"
]